# 08. Fixing the Messy Records

**Suggested time:** 45–60 minutes. Work through lessons in order; each notebook includes its own runnable examples.

All people, policies, prices, and records are fictional. Amounts use Philippine pesos (₱ / PHP). This is a coding lesson, not financial advice. Real coverage and claims depend on policy terms and applicable law. Never enter personal financial or health information.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mano-octavianojr/python-life-insurance-101/blob/main/notebooks/08_messy_records.ipynb)

Run this lesson in your browser with Google Colab. Sign in to Google if prompted. Use **File → Save a copy in Drive** to keep your work, then run the cells in order.

## Story opening

The club receives a synthetic CSV with missing names, negative premiums, and text where numbers should be. Maya and Leo must flag problems without inventing facts.

## Learning goals

Load a local CSV with pandas; convert numeric text; flag missing or invalid data.

## Insurance in everyday words

A policy record stores information about a policy. Incorrect records can mislead people. A missing premium must not be silently treated as zero, and a missing name must not be guessed. Flagging records for review is safer than claiming they are correct. This dataset expects strictly positive monthly payments; that classroom rule does not mean every real policy requires a monthly payment.

## Python toolbox

CSV is a plain-text table. pandas represents it as a DataFrame. `pd.to_numeric(..., errors="coerce")` converts numeric text and marks invalid text as missing. `.notna()` detects values that are not missing. `.fillna("")` replaces missing names with empty text for checking, not guessed names. `~valid` reverses the True/False selection. `&` combines column conditions; surround each comparison with parentheses. `.loc` selects rows.

## Predict, then run

Of six records, how many have a nonblank name and a strictly positive numeric monthly premium?

In [ ]:
from pathlib import Path
import pandas as pd
# Support starting Jupyter from the project root or notebooks folder.
data_path = Path("../data/policy_records.csv")
if not data_path.exists():
    data_path = Path("data/policy_records.csv")
# Colab opens a notebook without the repository's other files.
# If no local CSV exists, read the same public synthetic dataset from GitHub.
if data_path.exists():
    records = pd.read_csv(data_path)
else:
    data_url = "https://raw.githubusercontent.com/mano-octavianojr/python-life-insurance-101/main/data/policy_records.csv"
    records = pd.read_csv(data_url)
clean = records.copy()
clean["name"] = clean["name"].fillna("").str.strip()
clean["monthly_premium"] = pd.to_numeric(clean["monthly_premium"], errors="coerce")
valid = (clean["name"] != "") & clean["monthly_premium"].notna() & (clean["monthly_premium"] > 0)
accepted = clean.loc[valid].copy()
review = clean.loc[~valid].copy()
print("Accepted for this classroom check:")
print(accepted.to_string(index=False))
print("Needs review:")
print(review.to_string(index=False))

## Your turn

Calculate the total premiums of accepted records only. Count the rows needing review. Explain why filling every missing premium with 0 is misleading.

Try your answer in the cell below before reading the sample solution.

In [ ]:
# Student workspace: replace this comment with your attempt.
# This empty workspace is safe to run.

## Hint

Use accepted["monthly_premium"].sum() and len(review). Passing this check does not verify a real policy.

**Common error:** If loading fails, check the local data folder first. Without local data, the notebook needs an Internet connection to fetch the public GitHub CSV.

## Sample solution

Compare with your attempt after trying the challenge. There may be more than one correct approach.

In [ ]:
accepted_total = accepted["monthly_premium"].sum()
review_count = len(review)
print(f"Accepted monthly total: ₱{accepted_total:,.0f}; review count: {review_count}")

## Check your results

`assert` checks a condition. If it is false, Python raises AssertionError. These checks verify the supplied demos and sample solutions. If you intentionally change their inputs, update the expected results too.

In [ ]:
assert accepted["policy_id"].tolist() == ["P001", "P006"]
assert review_count == 4 and accepted_total == 950
assert len(records) == len(accepted) + len(review)

## More examples

Spend another 15–20 minutes on these optional examples. Run the earlier demo first, predict each result, then compare your prediction with the explanation. Use only fictional inputs.

### Keep a reason for each record flag

Predict whether a row can have more than one problem. Create separate Boolean columns so reviewers can see why a row was flagged.

In [ ]:
reason_flags = pd.DataFrame({
    "missing_name": clean["name"] == "",
    "missing_or_nonnumeric_premium": clean["monthly_premium"].isna(),
    "nonpositive_premium": clean["monthly_premium"].notna() & (clean["monthly_premium"] <= 0),
})
review_details = pd.concat([clean[["policy_id"]], reason_flags], axis=1)
print(review_details.to_string(index=False))
flagged_by_reason = reason_flags.any(axis=1)
assert flagged_by_reason.tolist() == (~valid).tolist()

**What happened and why:** .isna() identifies missing numeric values after conversion, including invalid numeric text. pd.concat(..., axis=1) puts columns side by side; .any(axis=1) is true if any flag in a row is true. These reasons reproduce the existing rule without changing records.

### Distinguish zero, missing, and text

Predict which values convert to numbers and which pass the strictly positive classroom rule.

In [ ]:
premium_examples = pd.Series(["0", "450", "unknown", None, "-20"])
converted_examples = pd.to_numeric(premium_examples, errors="coerce")
positive_examples = converted_examples.notna() & (converted_examples > 0)
print(pd.DataFrame({"original": premium_examples,
                    "numeric": converted_examples,
                    "passes_positive_rule": positive_examples}).to_string(index=False))
assert positive_examples.tolist() == [False, True, False, False, False]
assert converted_examples.isna().sum() == 2

**What happened and why:** Zero and -20 are known numbers that fail this rule. unknown and None become missing numeric values. Only 450 passes. Preserve the original text so missing and unparseable entries can be investigated separately.

## Discussion

Talk with a partner or write your reasoning in a Markdown cell before reading the suggested answers.

1. Why can the total number of flags exceed the number of flagged rows?

2. Does passing the name-and-premium rule verify the whole record?

### Suggested discussion answers

1. One record can have several problems. Count rows with any flag for a row count; summing all flags counts problems.

2. No. It does not check policy status, duplicate identifiers, or whether the supplied facts match an authoritative record.

## Story ending

Maya preserves the original file and separates four records for review. Leo explains that valid-looking numbers do not establish that a real policy is active.

## Exit ticket

Coding: What does missing numeric data mean? Insurance: Why should a missing payment amount not be guessed?

Write your answers in your own words in a new Markdown cell.